## Colab 준비
Colab에서 열었다면 **아래 칸을 맨 먼저 실행**하세요. 도우미 파일과 한글 글꼴을 받아요 (10~20초).

- 더 빠르게: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**
- 고친 내용 저장: **파일 → Drive에 사본 저장**
- 맥 실습실에서 열었다면 이 칸은 아무 일도 안 해요.

In [ ]:
import os, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/sangwon9401/silicon-descent/main/colab/"
    for f in ['common.py', 'solutions/__init__.py', 'solutions/sol00.py', 'solutions/sol01.py', 'solutions/sol02.py', 'solutions/sol04.py']:
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + f, f)
    os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
    print("Colab 준비 완료 ✅")
else:
    print("Colab이 아니에요. 그냥 넘어가요.")

# 00. 실습 준비: 파이썬과 PyTorch 기초

1단계 실습에서 쓰는 **모든 도구**를 여기서 하나씩 익혀요. 각 도구마다
① 무엇인지 설명 → ② 실행해 보는 예시 → ③ 아주 작은 연습 문제(자동 채점) 순서예요.

**이 노트북을 끝내면** 01번 실습의 `tok`, `x[..., :half]`, `x @ w.T`, `.view`, `einsum`, `masked_fill` 같은 코드가 다 읽혀요.

## 노트북 쓰는 법
- 회색 칸(셀)을 누르고 **Shift + Enter**를 누르면 실행돼요. 결과는 바로 아래에 나와요.
- 위에서부터 차례로 실행해요. 아래 셀은 위 셀의 결과(변수)를 써요.
- `...` 은 **내가 채울 빈칸**이에요. 파이썬에서 `...`은 '아직 아무것도 없음'이라는 뜻이라, 그대로 두면 채점에서 ❌가 나와요.
- 막히면 `# from solutions...` 줄의 맨 앞 `#`을 지우고 실행하면 정답을 불러와요.

In [ ]:
import torch
import torch.nn.functional as F
from common import *

print("준비 완료! torch 버전:", torch.__version__)

### import가 뭐예요?
다른 사람이 만든 도구 상자를 가져오는 거예요.
- `import torch` → `torch`라는 이름으로 PyTorch를 쓸 수 있어요. `torch.randn(...)`처럼 `torch.` 뒤에 도구 이름을 붙여요.
- `import torch.nn.functional as F` → 이름이 길어서 `F`라는 별명을 붙였어요. `F.silu(...)`처럼 써요.
- `from common import *` → 이 실습용으로 만든 `common.py` 안의 도구(`load_model`, `run_check`, `check_close` 등)를 **이름 그대로** 다 가져와요.

---
## 1. 파이썬 기본 5가지

**변수**: 값에 이름 붙이기. `x = 3`

**리스트**: 순서 있는 묶음. `[10, 20, 30]`. 0번부터 세요. `xs[0]`은 10, `xs[-1]`은 맨 뒤 30.

**딕셔너리**: 이름표로 찾는 묶음. `{"ln1": 표1, "wq": 표2}`. `d["wq"]`로 꺼내요.
01번 실습의 `W["layers"][0]["ln1"]` = "W에서 layers 꺼내고 → 그중 0번(첫 블록) → 그 안의 ln1".

**함수**: 입력을 받아 결과를 돌려주는 묶음. `def 이름(입력): ... return 결과`. **return을 안 쓰면 결과가 없어요(None)**.

**f-문자열**: `f"값은 {x}"` → 중괄호 안이 실제 값으로 바뀌어요.

In [ ]:
xs = [10, 20, 30]
print("첫 번째:", xs[0], "/ 마지막:", xs[-1], "/ 개수:", len(xs))

d = {"ln1": "정규화 가중치", "wq": "Q 가중치"}
print("d['wq'] =", d["wq"])


def add(a, b):
    return a + b


print(f"add(2, 3) = {add(2, 3)}")

# 리스트 컴프리헨션: 리스트를 한 줄로 만들기. "xs의 각 x에 대해 x + 1"
print([x + 1 for x in xs])

# lambda: 이름 없는 짧은 함수. run_check(이름, lambda: ...)는 '이 코드를 나중에 대신 실행해 줘'라는 뜻이에요.
f = lambda: add(1, 1)
print("f() =", f())

### 연습 1: 제곱 리스트
숫자 리스트 `xs`를 받아 각 숫자를 제곱한 리스트를 돌려주세요. `[1, 2, 3]` → `[1, 4, 9]`

힌트: `[x * x for x in xs]`

In [ ]:
def squares(xs):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import squares

In [ ]:
run_check("연습 1", lambda: check_equal("제곱 리스트", squares([1, 2, 3, 4]), [1, 4, 9, 16]))

---
## 2. 텐서: 숫자가 담긴 상자

PyTorch의 기본 단위는 **텐서(tensor)**예요. 숫자를 여러 차원으로 담은 상자예요.
- 숫자 하나 → 0차원, 줄 하나 `[1, 2, 3]` → 1차원(벡터), 표 → 2차원(행렬), 표 여러 장 → 3차원

**`.shape`(모양)이 제일 중요해요.** 실습에서 막히면 거의 항상 모양 문제예요. 막히면 `print(x.shape)`부터!

| 만들기 | 뜻 |
|---|---|
| `torch.tensor([1., 2.])` | 직접 숫자 넣기 |
| `torch.randn(3, 4)` | 무작위 숫자로 3×4 |
| `torch.zeros(3, 4)`, `torch.ones(3, 4)` | 0 또는 1로 채운 3×4 |
| `torch.arange(5)` | `[0, 1, 2, 3, 4]` |
| `torch.arange(2, 5)` | `[2, 3, 4]` (끝은 안 들어가요) |

꺼내기: `.tolist()` → 파이썬 리스트, `.item()` → 숫자 하나짜리 텐서를 그냥 숫자로.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.randn(3, 4)
print("a:", a, "모양", a.shape)
print("b 모양:", b.shape, " → 3줄 4칸")
print("arange(5):", torch.arange(5))
print("tolist:", torch.arange(3).tolist(), "/ item:", torch.tensor(7).item())

# reshape: 숫자 순서는 그대로, 모양만 바꾸기
g = torch.arange(6)
print("arange(6).reshape(2, 3):\n", g.reshape(2, 3))

### device: 어느 장치에서 계산하나
맥에서는 `mps`(맥 GPU), NVIDIA GPU면 `cuda`, 없으면 `cpu`.
계산하는 텐서들은 **같은 장치에 있어야** 해요. 옮길 때는 `.to(device)`.
다르면 `Expected all tensors to be on the same device` 에러가 나요.

In [ ]:
device = pick_device()
c = torch.randn(2, 2).to(device)
print("device:", device, "/ c는 어디에?", c.device)

### 연습 2: 0부터 11까지를 3×4 표로
힌트: `torch.arange(12).reshape(3, 4)`

In [ ]:
def make_grid():
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import make_grid

In [ ]:
run_check("연습 2", lambda: check_close("3×4 표", make_grid(), torch.arange(12).reshape(3, 4)))

---
## 3. 꺼내고 자르기 (인덱싱, 슬라이싱)

대괄호 안에 **차원마다** 무엇을 꺼낼지 쉼표로 적어요.

| 코드 | 뜻 (x가 [3, 4] 표일 때) |
|---|---|
| `x[0]` | 0번 줄 → 모양 [4] |
| `x[-1]` | 마지막 줄 |
| `x[-2:]` | 뒤에서 2줄 → [2, 4] |
| `x[:, 1]` | 모든 줄의 1번 칸 → [3]. `:`는 '전부' |
| `x[:, :2]` | 모든 줄의 앞 2칸 → [3, 2] |
| `x[..., :2]` | `...` = '앞 차원들은 전부'. 차원이 몇 개든 **마지막 차원**만 자를 때 써요 |
| `x[:, None, :]` | 가운데에 크기 1인 차원 끼우기 → [3, 1, 4] |

`a:b`는 a부터 b **직전**까지예요. `:half`는 처음부터 half 직전까지, `half:`는 half부터 끝까지.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print("x =\n", x)
print("x[0] =", x[0])
print("x[-2:] =\n", x[-2:])
print("x[:, 1] =", x[:, 1])
print("x[..., :2] =\n", x[..., :2])
print("x[:, None, :] 모양:", x[:, None, :].shape)

### 연습 3: 뒤 2줄 / 1번 칸
힌트: `x[-2:]`, `x[:, 1]`

In [ ]:
def last_two_rows(x):
    ...  # 여기를 채우세요


def second_col(x):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import last_two_rows, second_col

In [ ]:
_t = torch.arange(20).reshape(5, 4)
run_check("연습 3-1", lambda: check_close("뒤 2줄", last_two_rows(_t), _t[-2:]))
run_check("연습 3-2", lambda: check_close("1번 칸", second_col(_t), _t[:, 1]))

### torch.cat: 이어 붙이기
`torch.cat([a, b], dim=-1)` → a 뒤에 b를 **마지막 차원 방향**으로 붙여요. `dim=0`이면 아래로(줄 방향) 붙여요.
`-x`는 모든 숫자의 부호를 바꿔요.

In [ ]:
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[5, 6], [7, 8]])
print("dim=-1 (옆으로):\n", torch.cat([a, b], dim=-1))
print("dim=0 (아래로):\n", torch.cat([a, b], dim=0))

### 연습 4: 마지막 차원의 앞 절반과 뒤 절반 바꾸기
`[1, 2, 3, 4]` → `[3, 4, 1, 2]`. 차원이 몇 개든 마지막 차원에 대해서요.

힌트
- `half = x.shape[-1] // 2` (`//`는 몫만 남기는 나눗셈)
- 앞 절반 `x[..., :half]`, 뒤 절반 `x[..., half:]`
- `torch.cat([뒤, 앞], dim=-1)`

(01번 실습의 `rotate_half`가 이것과 아주 비슷해요!)

In [ ]:
def swap_halves(x):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import swap_halves

In [ ]:
_r = torch.randn(3, 2, 8)
run_check("연습 4", lambda: check_close("절반 바꾸기", swap_halves(_r), torch.cat([_r[..., 4:], _r[..., :4]], dim=-1)))

---
## 4. 계산하기

**원소별 계산**: `+ - * /`, `x.pow(2)`(제곱), `x.sqrt()`, `torch.rsqrt(x)`(1/√x). 같은 자리끼리 계산해요.

**줄이기**: `x.sum(dim=-1)`, `x.mean(dim=-1)` → 마지막 차원을 합/평균으로 **없애요**. [3, 4] → [3]
- `keepdim=True`를 붙이면 없애지 않고 크기 1로 남겨요. [3, 4] → [3, 1]. 그래야 원래 [3, 4]와 나눌 수 있어요.

**브로드캐스팅**: 모양이 달라도 크기 1인 차원은 자동으로 늘려서 계산해 줘요.
[3, 4] ÷ [3, 1] → [3, 1]이 4칸으로 복사된 것처럼 계산돼요.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0], [2.0, 2.0, 2.0, 2.0]])
print("x.pow(2) =\n", x.pow(2))
print("mean(dim=-1) =", x.mean(dim=-1), " 모양", x.mean(dim=-1).shape)
m = x.mean(dim=-1, keepdim=True)
print("keepdim=True 모양:", m.shape)
print("x / m (브로드캐스팅) =\n", x / m)

### 연습 5: 줄마다 제곱의 평균 (keepdim 유지)
모양 [T, D] → [T, 1]. 01번 실습 RMSNorm의 첫 단계예요.

힌트: `x.pow(2).mean(dim=-1, keepdim=True)`

In [ ]:
def row_mean_sq(x):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import row_mean_sq

In [ ]:
_x = torch.randn(5, 576)
run_check("연습 5", lambda: check_close("제곱 평균", row_mean_sq(_x), (_x ** 2).mean(-1, keepdim=True)))

---
## 5. 행렬곱 `@`와 `.T`

`a @ b`는 행렬곱이에요(0단계 3번 강의). **a의 마지막 크기 = b의 첫 크기**여야 해요. [T, 576] @ [576, 192] → [T, 192]

HuggingFace 가중치는 모양이 **[출력, 입력]**으로 저장돼 있어요. 예: Wk는 [192, 576].
그래서 x [T, 576]에 곱하려면 뒤집어서(`.T`, 전치) [576, 192]로 만들어 `x @ w.T`로 곱해요.

모양이 안 맞으면 이런 에러가 나요: `mat1 and mat2 shapes cannot be multiplied (5x576 and 192x576)` → `.T`를 빼먹은 거예요!

In [ ]:
x = torch.randn(5, 576)
w = torch.randn(192, 576)   # [출력 192, 입력 576]
print("w.T 모양:", w.T.shape)
print("x @ w.T 모양:", (x @ w.T).shape)

### 연습 6: 선형 층
x [T, in], w [out, in] → [T, out]

In [ ]:
def linear(x, w):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import linear

In [ ]:
run_check("연습 6", lambda: check_close("선형 층", linear(x, w), x @ w.T, atol=1e-3))

---
## 6. 모양 바꾸기: view, reshape, repeat_interleave

`x.view(T, 9, 64)` → 숫자 순서는 그대로 두고 모양만 [T, 576]에서 [T, 9, 64]로. **576 = 9 × 64**라서 가능해요.
576칸을 '64칸씩 9묶음(헤드 9개)'으로 보는 거예요(0단계 6번 멀티헤드).

`-1`을 쓰면 '나머지는 알아서 계산해'예요. `x.view(T, 9, -1)` → 마지막은 64로 자동.
`reshape`는 `view`와 거의 같은데 더 너그러워요(메모리가 흩어져 있어도 동작). 헷갈리면 `reshape`를 써도 돼요.

`k.repeat_interleave(3, dim=1)` → 1번 차원의 각 칸을 3번씩 복사해요. [S, 3, 64] → [S, 9, 64]. GQA에서 KV 헤드를 Q 헤드 수에 맞출 때 써요.

In [ ]:
x = torch.arange(12).reshape(2, 6)
print("view(2, 3, 2):\n", x.view(2, 3, 2))
h = torch.tensor([[10, 20, 30]])              # [1, 3]
print("repeat_interleave(2, dim=1):", h.repeat_interleave(2, dim=1))   # 10,10,20,20,30,30

### 연습 7: 헤드로 나누기 / 헤드 복사하기
- `split_heads(x, n_heads)`: [T, D] → [T, n_heads, D / n_heads]. 힌트: `x.view(T, n_heads, -1)`
- `repeat_heads(k, times)`: [S, n, d] → [S, n × times, d]. 힌트: `k.repeat_interleave(times, dim=1)`

In [ ]:
def split_heads(x, n_heads):
    T = x.shape[0]
    ...  # 여기를 채우세요


def repeat_heads(k, times):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import split_heads, repeat_heads

In [ ]:
_h = torch.randn(4, 576)
_k = torch.randn(4, 3, 64)
run_check("연습 7-1", lambda: check_close("헤드 나누기", split_heads(_h, 9), _h.view(4, 9, 64)))
run_check("연습 7-2", lambda: check_close("헤드 복사", repeat_heads(_k, 3), _k.repeat_interleave(3, dim=1)))

---
## 7. einsum: 이름표로 쓰는 곱셈

`torch.einsum("규칙", a, b)`는 **차원마다 글자 이름을 붙여서** 곱하는 방법을 적어요.

읽는 법 3가지
1. 쉼표 앞은 a의 차원 이름, 뒤는 b의 차원 이름, 화살표 뒤는 결과의 차원 이름.
2. a와 b에 **같은 글자**가 있으면 그 차원끼리 짝지어 곱해요.
3. 화살표 뒤에 **없는 글자**는 더해서 없애요.

예: `"ij,jk->ik"` = 보통 행렬곱 `a @ b`. j가 결과에 없으니 j 방향으로 곱해서 더해요(내적).

01번 실습의 `"tnd,snd->nts"`
- q는 [t(내 토큰), n(헤드), d(64칸)], k는 [s(상대 토큰), n, d]
- n은 양쪽에 있고 결과에도 있음 → 헤드끼리만 짝지음(헤드마다 따로)
- d는 결과에 없음 → 64칸을 곱해서 더함 = **q·k 내적**
- 결과 [n, t, s] = 헤드마다, 내 토큰 t가 상대 토큰 s를 보는 점수표

In [ ]:
a = torch.randn(2, 3)
b = torch.randn(3, 4)
print("einsum ij,jk->ik 와 a @ b 가 같나?", torch.allclose(torch.einsum("ij,jk->ik", a, b), a @ b))

q = torch.randn(5, 9, 64)   # 토큰 5개, 헤드 9개
k = torch.randn(5, 9, 64)
print("점수표 모양 [헤드, 내 토큰, 상대 토큰]:", torch.einsum("tnd,snd->nts", q, k).shape)

### 연습 8: 헤드마다 점수표
q [T, n, d], k [S, n, d] → [n, T, S]

In [ ]:
def head_scores(q, k):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import head_scores

In [ ]:
_q, _k2 = torch.randn(3, 2, 8), torch.randn(4, 2, 8)
run_check("연습 8", lambda: check_close("점수표", head_scores(_q, _k2), (_q.transpose(0, 1) @ _k2.permute(1, 2, 0))))

---
## 8. 비교, masked_fill, softmax

**비교하면 참/거짓 텐서가 나와요.** `torch.arange(4) > 1` → `[False, False, True, True]`

**브로드캐스팅으로 표 만들기**: `[:, None]`(세로)와 `[None, :]`(가로)를 비교하면 모든 짝의 비교표가 생겨요.
`qpos = arange(T)[:, None]`, `kpos = arange(T)[None, :]` → `kpos > qpos`는 '상대가 나보다 뒤(미래)'인 칸만 True인 표예요.

**`x.masked_fill(조건, 값)`**: 조건이 True인 칸을 값으로 바꿔요. 미래 칸을 `float("-inf")`(마이너스 무한대)로 → softmax 후 0%가 돼요(0단계 5번).

**`x.softmax(dim=-1)`**: 마지막 차원을 합이 1인 비율로(0단계 4번).

In [ ]:
T = 4
qpos = torch.arange(T)[:, None]
kpos = torch.arange(T)[None, :]
future = kpos > qpos
print("미래인 칸 (True):\n", future)
s = torch.zeros(T, T).masked_fill(future, float("-inf"))
print("가린 점수:\n", s)
print("softmax:\n", s.softmax(dim=-1))

### 연습 9: 정사각형 점수표의 미래 가리기
scores [..., T, T]에서 '상대가 나보다 뒤'인 칸을 `-inf`로.

In [ ]:
def causal_mask(scores):
    T = scores.shape[-1]
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import causal_mask

In [ ]:
_s = torch.randn(2, 5, 5)
run_check("연습 9", lambda: check_close("미래 가리기", causal_mask(_s), _s.masked_fill(torch.ones(5, 5, dtype=torch.bool).triu(1), float("-inf"))))

---
## 9. 나머지 도구들

| 도구 | 뜻 | 어디서 |
|---|---|---|
| `F.silu(x)` | SiLU 활성화 (0단계 10번) | 01 MLP |
| `x.argmax()` / `x.argmax(dim=-1)` | 가장 큰 값의 **위치(번호)** | 다음 토큰 고르기 |
| `logits[-1]` | 마지막 토큰의 점수 | 다음 토큰 고르기 |
| `with torch.no_grad():` | 학습 안 하니 기울기 기록 끄기 (빠르고 메모리 절약) | 02 |
| `x.sort(dim=-1, descending=True)` | 정렬. (값, 원래 위치) 두 개를 돌려줘요 | 04 top-k/p |
| `x.cumsum(dim=-1)` | 누적 합 [1, 2, 3] → [1, 3, 6] | 04 top-p |
| `torch.where(조건, a, b)` | 조건이 참이면 a, 아니면 b | 04 |
| `torch.multinomial(probs, 1)` | 확률대로 하나 뽑기 | 04 |
| `x.clone()` | 복사본 만들기 | 01 |

In [ ]:
logits = torch.tensor([[0.1, 2.0, 0.5], [3.0, 0.2, 1.0]])
print("마지막 줄:", logits[-1], "→ argmax:", logits[-1].argmax().item())
v, i = torch.tensor([3.0, 1.0, 2.0]).sort(descending=True)
print("sort 값:", v, "원래 위치:", i)
print("cumsum:", torch.tensor([1, 2, 3]).cumsum(dim=-1))
print("where:", torch.where(torch.tensor([True, False]), torch.tensor([1, 1]), torch.tensor([9, 9])))

### 연습 10: 다음 토큰 번호
logits [T, V]에서 **마지막 토큰** 줄의 가장 큰 값 위치. 힌트: `logits[-1].argmax()`

In [ ]:
def next_token(logits):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol00 import next_token

In [ ]:
_l = torch.randn(6, 100)
run_check("연습 10", lambda: check_close("다음 토큰", next_token(_l), _l[-1].argmax()))

---
## 10. 토크나이저 `tok`

01번 실습의 `model, tok = load_model(device)`에서 `tok`는 **토크나이저**예요(0단계 1번). 문장 ↔ 토큰 번호를 바꿔 줘요.

| 코드 | 뜻 |
|---|---|
| `tok("The capital", return_tensors="pt")` | 문장 → 결과 묶음. `return_tensors="pt"`는 'PyTorch 텐서로 줘' |
| `.input_ids` | 그 묶음에서 토큰 번호만 꺼내기 → 모양 [1, T] (1은 문장 개수) |
| `[0]` | 첫 번째(유일한) 문장만 → [T] |
| `tok.decode([번호들])` | 번호 → 글자 |

그래서 `ids = tok(prompt, return_tensors="pt").input_ids[0].to(device)` =
"문장을 토큰 번호 텐서로 바꾸고, 첫 문장만 꺼내서, 계산 장치로 옮겨라"예요.

아래 셀은 토크나이저만 내려받아요(몇 MB). 인터넷이 필요해요.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
enc = tok("The capital of France is", return_tensors="pt")
print("묶음 안에 든 것:", list(enc.keys()))
ids = enc.input_ids[0]
print("토큰 번호:", ids.tolist())
print("조각마다 글자:", [tok.decode([i]) for i in ids])
print("다시 문장으로:", tok.decode(ids))

---
## 🎉 다 했어요!

막혔을 때 쓰는 3단계 습관
1. **`print(x.shape)`** 로 모양부터 확인해요. 에러의 대부분은 모양이 안 맞아서예요.
2. 에러 메시지의 **마지막 줄**을 읽어요. `shapes cannot be multiplied` → `.T` 확인, `same device` → `.to(device)` 확인, `NoneType` → `return` 빠뜨림.
3. 작은 숫자로 따로 실험해 봐요. 새 셀을 만들어 `torch.arange(6).reshape(2, 3)` 같은 걸로 시험해 보면 금방 보여요.

이제 **01번 실습**으로 가세요. 거기 나오는 코드가 이 노트북의 도구 조합이에요.